# NEXUS-SLM: Clean-Data LoRA Fine-Tuning Pipeline v2 (Qwen2.5-0.5B-Instruct)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ASstyles/NEXUS-SLM/blob/sprint/clean-data-model-improvement/notebooks/clean_data_qwen25_05b_lora_training_v2.ipynb)

**Sprint:** `sprint/clean-data-model-improvement`  
**Target Candidate:** `qwen25_05b_clean_v2_lora`  
**Base Model:** `Qwen/Qwen2.5-0.5B-Instruct` (494M parameters, 151,643 vocabulary)  
**Seed:** `42`  
**Dataset Invariant:** Strict template-grouped split isolation (0% text/template leakage)  
- **Training Split:** `data/v2_clean_template_grouped/train_clean.jsonl` ($N=4,797$)
- **Validation Split:** `data/v2_clean_template_grouped/val_clean.jsonl` ($N=449$)
- **Locked Test Split:** `data/v2_clean_template_grouped/test_clean.jsonl` ($N=746$) — **STRICTLY QUARANTINED** (never used in training or model selection)

---
### Quick-Start Free Cloud Execution Instructions
1. **Google Colab (Recommended - Free T4 GPU):**
   - Click the **Open In Colab** badge above or open [Google Colab](https://colab.research.google.com) and upload this notebook.
   - Go to `Runtime` → `Change runtime type` → Hardware accelerator: `T4 GPU`.
   - Click `Runtime` → `Run all`.
   - When complete, `qwen25_05b_clean_v2_lora.zip` will automatically download via browser prompt.
2. **Kaggle Notebooks (Alternative - Free T4 x2 or P100 GPU):**
   - Go to [Kaggle Code](https://www.kaggle.com/code) → `New Notebook` → `File` → `Import Notebook`.
   - In `Settings` → `Accelerator`, select `GPU T4 x2` (or `GPU P100`). Enable `Internet: ON`.
   - Click `Run All`. Download `qwen25_05b_clean_v2_lora.zip` from `/kaggle/working/`.
3. **Estimated Execution Time:** ~16–20 minutes on Nvidia T4 GPU.

## 1. Hardware & GPU Preflight Check
Verifies CUDA availability, active GPU name, and VRAM. Aborts cleanly if running in CPU mode to prevent out-of-memory thrashing.

In [ ]:
import os
import sys
import platform
import psutil
import torch

print('=' * 75)
print('NEXUS-SLM: GPU PREFLIGHT DIAGNOSTICS')
print('=' * 75)
print(f'Platform:         {platform.system()} {platform.release()} ({platform.machine()})')
print(f'Python:           {platform.python_version()}')
print(f'PyTorch:          {torch.__version__}')
print(f'CUDA Available:   {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError(
        'FATAL: CUDA GPU is required for Qwen2.5-0.5B LoRA training.\n'
        'On Google Colab: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU.\n'
        'On Kaggle: Settings -> Accelerator -> GPU T4 or P100.\n'
        'CPU execution is disabled to prevent memory paging and system freezes.'
    )

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
free_vram_gb = round(torch.cuda.mem_get_info()[0] / (1024**3), 2)
bf16_supported = torch.cuda.is_bf16_supported()

print(f'Active GPU:       {gpu_name}')
print(f'Total VRAM:       {total_vram_gb} GB')
print(f'Free VRAM:        {free_vram_gb} GB')
print(f'bfloat16 Support: {bf16_supported}')
print('=' * 75)
print('Hardware Preflight PASSED!')

## 2. Dependency Installation
Installs verified Hugging Face and PEFT libraries without clobbering pre-installed CUDA PyTorch.

In [ ]:
# Note: PyTorch is pre-installed in Colab/Kaggle with matching CUDA drivers. We install HF packages safely.
# Remove stale/incompatible torchao to prevent PEFT dispatcher collisions, and quote package specs
!pip uninstall -y torchao 2>/dev/null || true
!pip install -q 'transformers>=4.46.0,<5.0.0' 'peft>=0.13.0' 'accelerate>=1.0.0' 'datasets>=3.0.0' 'scikit-learn>=1.5.0' 'safetensors' 'psutil'
print('Dependencies successfully installed!')

## 3. Dataset Setup & Cryptographic SHA-256 Verification
Self-extracts the canonical verified clean training ($N=4,797$) and validation ($N=449$) datasets.
Strictly quarantines test data to guarantee zero leakage and prevent post-hoc bias.

In [ ]:
import base64
import gzip
import hashlib
import json
from pathlib import Path

WORK_DIR = Path('/kaggle/working/nexus_training') if Path('/kaggle/working').exists() else Path('./nexus_training')
WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = WORK_DIR / 'data' / 'v2_clean_template_grouped'
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_file = DATA_DIR / 'train_clean.jsonl'
val_file = DATA_DIR / 'val_clean.jsonl'

# Embedded compressed payloads
EMBEDDED_TRAIN = '...'
EMBEDDED_VAL = 'H4sIAG/e5GoC/+2dXY/j1nnH7wv0OxwvetECmy153tmbgRMn8cJwbWRdFL0SzkjcIVcSJZPUrqdBAMdB7QBBUASFY6A3hQGNYThuYgRoEN/YX0UfpedQM7tDjbQzI1Ez58z+1zBGoiiK5O88r3zOc35+Lx/c+ydy7yntRZFgSXzvPrlXpx/UbuPrZZkek8PM5OSnpcnMkFTpU0OGaTEo7R9DsnQ0HeVFSorZ+DAtyfDYbjP5gTtI39Tp0aQ8dgc6Sou0NKPe+7PUbrAfTst8UuZ18+E4HeSzsds6MsXRzBylbmuWF0ejvMrc9jJ9f5aXadXLZmNT2E8fm1GVNoeZPE0LU/TdV35+r5rMyublvf6stD8/6I1nozq3Z2gPO+rV6Xg6spt71XFRZ2mVV+7gA1ObXp2npfte1c/SsemZYtAbpR/k/UnRe5qW+eM8Hbh9l9dR53bzOK2zSXPrpqau07LoVaNJ3RtPymk2GU2Omst8akb5YLl/VZt6VjU3o5zM7PHrclZnvdM9loevTXmUutN+bPLRrEzt0QbN5RSTIr33i1/87d/8/Dwumii2Ade7pqqeTcoBsXctrYm9A0NiLywfkamFZbKcGEMcUofrPpmOUlOlxF58f2i5lpMHqwSnpwfsNQdcRZjlR9m1ANqLB7840eo8vzfT0WhC6tSM75P3StMf2usmlSmG2bGTNjNekrN8HLrcobvAySJKS3tuy29D1LpCFbc143lU77hbTvru1owcnMHkVJ7Ws1nuCDLXJmPPw93+XjU1ZeVu20VKkm6k9EY6yu1VWP04OSblbJCSOjOrhAanO/XyqpqlYLQHRpEUmxllpiDPjLU1jRFapVOmj935uJucVhds0GjyDGh2QsOEiDah+emj98hhPrIKzrkOT01lbdKwNmttkNtvjfGBk/ByQJU9qaLO+z17ueXITHuDvDLjw9xeljv4RV6Ctnk9LNwpNjubEWl0mdV3FlY+MoejtAuvPIVAXUOgBE1agN6yfpwxU/K2jZaskz5OyeFsNCSNX0DObA9x9y+39smqwDol2eR63KAGt/HvOGctq/R26gLczF4Pqez1D6ssr0mx9LnvWx1Y5cRGvaYwgywf2lt/ynCYH69XiY9LM3OWazopa7gVXcW/grXE61170YUh9vdfZCQO7aVYGSJH5vg01jV9+/O1/dNPK+dn5DbwHeej9MhcEDS7j9u3t9wX9qybpAXT56E9Su1l2ztKGqfO/mhFliffb06BuKNWZGD3scGw9QyflQPSt37iUeoUpeNyzVwFDNuWKjKRogUuL8/yRj/+YJpbw+WOTIp8aJ2PfjYbNsqSPJk4ToNZvz6zcCSbrVeSp/v1+hN35/ICaaZOwFnHXm8A97Zxyds33/jJj8g/p7XVlcuUk7VsZpBW2fPk0zs/e+MHcaRpxMgwJaP82Hou2eyotuFaY/JsKP2AWJuZOn07bAxn6g4TxzGPbbQ9NcfOn6mdGk4de/e9op6d/Y69gY/z8jS/dTY8yFtlPnUHd2q9Jnk1tMbYjK3JdT9ryjQv1oj+8Ti1Gvv0zmAA7TkE4VTRDWPr3TIdW0Vr31pcjRleF9NXs8Oqbzk3h4djtIeg3kI6T+jfjCnJ69Ppc8/o7CnAKpk67WeFtcKj9ckwxBlb5ZGlEhdoPE9NOu1WHSMreQuZY67UBS4/NfZyn8fl16MC8dhZcSWaXUDy6Jy5sCFcbQmQQW79jedqzMYIJXlcpumLhIr1Gx6b44FZuhc21mt2fgBTdCPPONvPqBfzPy5Ofr2Yf7SYf7k4+YQs5h8v5l/YV4v5Xxo3M4kSa7Ls9k8X8+/sn98t5t8sTn67mH9uXxC37/z7xfxPyw9P7J/FybzZ++RXi/lXzz/5ejGfuw/n/2l/a3m070nz6bdn6bfmVD5Zfrx89c3yjL5y3z179c3plxfzv5KzY9r9PlzMf798/+lyv28XJ79sLuu/m0N9tjyz6z2eXa80BjnGV6Nvn5rCHJky79WTYVr00qLM+5lz99fEPHHcLo0YN0mfyWPyd5pTbXXDYW5/xCqKyZiMj5dhy9PckEc/fEj+5d2H5HBWk9NgglTZ5FnVJCLsN6zeWeb63GCljDKlrxuFWCVTO8fqOjkIBCIvD0SsohEbgDMuVVfAE5oIdm3gSDl1zptGSSvw/NcsLcgz94DyBSxOYxEzUqamnznmo8lpTjEvyI8yewtyq02ygy5KaMDy0rxvxNSlwHgso66Brcv5QNNeXpqmL6XFFJV8Pa2li3Wws/sDubqcFI8vI0WZ1pqC1C3n4WPWysQtk+/uaUlpTE3iWMaRi1XPvJCzzDp5+CP7H/lhk1l3ZWv2hI6yB8sNVUoevf3IVfM2Tzhddc7S8xmkzTOX1qPPUWphugoEexgznTah8nKULC/3zOVpR9QvEvGlKSoXdNuR09xmF1qX9ljLY5wl8ZepeWK/sCzYsqPM+rvuiQCS9n74yiyOXzIQE8oxDK8etWEo7jQUtXjZUIwjibEIlbj3cRhrHl3mRSVacq7We1FvHtu9zKEZwJG6AZdXcXkpLEE1TbqFhdB/S16aXh76S8E5BS8vKr1oO/h/aLH00/ypNYOmaGyrNcH1+fgyVkwx8sxUrtqjfy7NelpBSWaVKywau0fc7jBNNpb0TTl4QB6SInVVYaffrLO8Wh76PikmJRlMSPP7k6Mi/3dn6lNSZfl06o5nBoPSFXHaMfJGOsqsWX53WXvinv/Znc+dQD52o8HekNHxtYpzkT3aZgDJdlXQpQNICE2tp4cBhAG0HEB2PFxrAEkWRcnd1EB4Wri1FuL8WoPIOiCCRqEMooOD116DItr7U6v2jOJLy1diqkXC91q+8kNLzdhzn6GC5Q5XsMRJcr2RJ0XM5Ks78jZW6GHwXX/wWWecXmvwqVjJCIMPg68Lzafb6YfLS0a5YoJi8GHwdVA4yttV5z9bPq85fRi0LAxs5ie6n3K1gWY6vd8UD54VFbpd03NlhMWE9EuLqH6wcz8TpC53edQoI+0HWeSlu4e7Mn8HYntnyOp2BA6xvVM6WVCI7d0kqxjE9q7CFe1ZzIB7p2ZSqXbTsrNmHtOmVdLzLh7LyrmzNiBP82nmms61y+RQYulDiWUCnndKPoW+JZ6YBb+XCmgOnHcIp0yA8+7gtOoWxvMO8aSMdsXT/sO0jdsWz1vTtmDZvSPU8mtdU47z06uqetYfuhq5aVoMmjaVp+mG3N5LcpxW9ooG5vg+GU+K9Ph0slc66IIryuR20bgJ9wArqmj3oX0p8wAtcoB7EFrJILR3VWgVzOxdxMqhi+9oCoJBYO9guKMSmNi7Gskm7QLm/23KfD87qwT+kizm/9dUB/9hMf8zOSsD/ri7DOGG0t9XjNwVViOMI7YVqnWz74Br77hWun7uS7I2JPeA6aqYpLwJTJj5sDMpxgQEyn9MNIqBKQBMXAJTAEpPcGDyHhON221KX1+uFdd06tBSCdcs95DUZkjGbuW5qXv069o3Tk02K+xFr1nQc5vmBUhdXdbdTfINmISKVSK7xoQ6p61JxckGUom0nwkIlBc9Z2IabcAkkziJBATKn977dAMpRrViMUj504J0ky8hIypFDNXnR7stvslCqThSAr6EPz2Yqdjk9SVJLDVIeSNTidzk9QlLEUbKH8dPtYzUO++9u2lJ4NN+kD37N60qENhH+oEJCRre0BAxaPhDgyfQVB7RiBVo+KOphOiEBkLI3QUjioHCE6lozxU5h2JdKRVw7Nu5TRKYDF9oiKi9Mug7z5ck0HEkpbaRORnMpqPccfnHwgxHeUPLTWlNn5CBWxXvxZJ279vbZX+GHKZDY+P4tXNby4krF+41TZpMXtSY3dpFAG85xus5xpxxzm6eIzTlViuHqESv56ikjhIKeQxFHles3At5TKysKnAMhWMcrefIuVY0vnmOcHu2Va0J3eDqMKZZzIEyIKkUbANKKSWHlQyGY3uxvRccZRIrDpEMylCq9ShdgZ+kCEBCCUBWcsgvVkpNuOY7O65rZy9Cu+4DpVsMd1OhjGBaMTI0ZJBZdM/M1FTLIoxh08fMQj7F+SQ/TndvaA/B22lWt4o3FXtSrhOd7AkkjOE+VifYWGKtKI+5AMuQWOoNLHXEmdJgGdIaXZt0rBQyYQosQ1rbZ1OFMNVaRAwsA2K5uYaYRQnlYBnQ4j0bK/cld8v4IiAJZc1DuXF+dBIpSvcEEnmAzrUr3xSNaCkYh3YNKUvANs6GlyxqpoReieW6FB0U7I2yZFK2opEff1CXhlTZbDQkw+PZhWlRh/lohPlQ+ymRY1wosPCEhWBJByygr3YGoRgDCC8kot3wENrpNlm0805bskCUszMI2e7fBe10e/ZaAIQX9lopgPBCNbWrUGGvb5OFTsDCExYrE9Fen05JvzRVRiYFsRfVH05m9SqQOu1nhWts38uraoYlYfbBhbYzUeDiCZe4vf5oF1zWWHdAuZ6wRIDiHRSmY0DxDkq7xRXMii9mZaVEYx2Xg4MDyMtNu8cJhxLzzzfWUGI+PrFtz0WDsHghLELtwbJAXjqJJUXX8oJFW3f2kCMYFw+5cNFO4L9tz7MpozOksvdiWGV5TQqT5csZsMO8ykmRElOYQZYPLQnS9LC2HywnyV6onHxcmpmrtJtOShTAdjOnWbQXRwMy/5Hxdv+kG0X2kuVyge0SbBrKMTBkMoJyDA4ZlGOIbgjtVNLWBMsAtxdwTHQqb69dnF8DJdm9kmQadi0w7/EWQ2uox+0lrb0WJSQtBEljQBaY96g0lGOAayCq6AK2qbG3YETMoaU1NG5KtnPnyXunyxpaRuSJKYamzEk/s0jXzK/HQoj7Wqg8vnlc0IjbsWrPaYFoeb5kuaDAFdKC2Bq4AlKGEQOugKSL3oLtwvTN7URLaviEoSxkzhBuheQTyi7CrTUPVKAI90BLrcnPQ7j8FS4tgSuk1BOHSxgIK5q0chlvmLG9eQOLqz90d3EFwSAd5fZijlGVva+qbOv2ic6AYK7PrjRWFjmBeNw2kJUugRCP21VWbUsP8bhtIJxLAPHKgEQKQLwyIIrCgPhDo73w2E40MFN3ZxrtuWpQVrcORCgoK39oxALi4VX40aH1gHjsHHvIl9FYtxgPRGT/TR7aqw7+xNVGk1FakIH9P5s1FdTXKaDGU5GtZyq0+wKDxK2RUO0Ks51JYFLB9hNSdbcoUGy0HYYoAgYPZkbpy431Oj8KKLpH0bG1BoYtTbWEqfZlpq2ERHhgqrkCBh/sQ4JozpOeiBAILyw1BQYPWpgoSIMHZbVR2zws5p8uTn61mH+1mP9pMf+OLOYfL+ZfLE4+Wcz/Yt/8dTH/aDH//eLkt6efu7XudUKjJCbNV39j/8wXJ79czL9pdp2T5tUXzTe+aPax793hPrpv3322mP+5+cKHzcFPPrF/ftfsYXf/3L6w779dHvWz52d0Yv8sTubrq677juToioNjkGNkuIcbT01hjkyZ9+rJMC16aVHm/WycFvW6Omyhdx0wQsecRV4PmPVF+hvGy6vm31xvwMTRzhpGcmu2qdcDZqNbjDGzxZiJ2guIbTNmYjtoKMbMKzNmIi13HjOJpoxreDKviCcTy51dX8GYpPBkXg0Nkyi+u+srYgkN86q4vnrnAZMwexiJAfOKmCQudh4wXEQ6gUl6RUwS3V3DxImkMQKlV8jv3TmDp7liWmDMvDp6hu8cKykW2fAanswrksGTaucMXsy4UFAyr46S2T3rmyQuI4Mx88qMGcFWxowl/P1i/oeG+OeW0JcNtQ/te/vmj82HFvN3q8TK9LE7UXf306oGs67nd7mFKm8AFCz4jpQSDnEKAdTKDAyA8hWUTBhABSFREUCFIVHsJmwUktQ7UpIJKPlPKZag5L9XrhVMUxg+BEAFovcAKgxQqxURMFAeUtKUQpzCaETXip0eGXunlmsYj/OROTg46KLzGSYDX7MfnZAvgQIit9EunoKIZ0QgI7510Yw7I4I1kTqw7N3hwFz6DtbjYdBXnjXFphAQj/QVh/nwCIeKNKTDp8X1gMMnHELDmvsVfbQbAV9IZHXTwx9UrmvUBeTEL6+XMbhZ3uCgWraCkLfKfHpsyDgtUzI1xZN8aO81sdeWj9xC1Pb/qno2KQfE3ta0JvYWDcl0ZvGRwyx9sm7t6bNv9JpvQKY66pNnwfHbBAfR24pa0najb5oaPO+tqMl2bTuUZDDg2h2OoSQDMW0xxC1MPRlD3MLzSSjELUxxi2M4k+FZN6UhbmHqSQHrFiA1BXELEZyiGuIWnrhp5JPDdCYV9GSYepJFABcgOKbilQnpXy+nITUTkpouYPbVf7gOYG7zcgLTp03zL9c47KvFya+XbcFWeR3mI3fPMKWs0yelTFJxk7jQOGo3XBdmPm+J64H9Bwm7GQmT7CYlDLM3dyamuQKxsIi1vUU4HX7j4orC6QgJFyxYWMSU1tCHAXn1iYCAhUUs5iAWVhgmKIiFZcTiCF5iQF4iVcAVkAVj0IehhWHwEgOzYLI1w+J1M0irbLlKVyQ0o2SYkrx4Osn7KRmnpMjLQWbcg80qm42GpEqJGWT5kJSmynLyZDLIyZE5zklm8ge7qUmUM55SrOxJFXXe79nLLUdm2hvklRkf5vay3MHXiWF7quc5qFxqpfQNQMXT6T1wXSkVOceVsUSr+Aa4ojqrc2HVm6BKSSVlENZAubbLzc8LaxLpCEo4UK6abzKuWlIaCXANlCtNNskrj1Syi7y+drEHDczrjWGlm7AqziOY10C5KrZJXDlnMQPXUOV1Y+zKVGTR7sB1XS8woL1BkdVyo8iqKIHIBspVJJtENqFK8QRcA+W60SOOhZACEWywGadNLnHCKBMRuAaaHo7EJtdJ8JgmSA+HKKx8E1StE02RbgpVWDcaVxHxmHEIa4BQ6YplnU4du8PZkcNyn/RNWbsNub0nFt9g2X43m1iImSFmVpK0LCclEVFEjFluXcezTvtZkfftzUW/5E5nBkdcgl/A/Gi7yyv4hSZ/gnnAD3ZxO+FLKOAFK3nt+T3QnIHxizksX9D82jNJwC8wfrzd9QTGLyzhg/IMmR9D2BCw8Al4nkHziySEL1jNqZHwDJrfylMk8AvN84TyDDdnxiB8AfMTkrbChrcyeynk0J48GZqyMKSfmcxuySaTosFlpnaju17i9mzvcIGa6ffteda9/mhS2XOByHWCbHUZx5tEhnnq20mZkLeGDLpxW2orfSGgG0MQNAFkYSETGroxPGpaJKAWHjUF9RicoDF4+4Ehi4EsNGQ8ArLQ/EZkrkILqBNkroLLD2tIWWCKEVIWnGJUDNF0eA9ibjHZCGrbLlnLVCtz9bY9NzJs0JlnppwNMvL3o0l/mA7+gQxy+8Hzh9L3yXC5vO3Qnmuab0Zm/6ZVBWvWDS4qbwUXBGxbYpx3JGBrWka+nBm8j+1ELEogYkERi5MIxMKSsZiCWFDEaHs2KYj5T0zAVQzMVZQcsVhAbkd77RXg8ly6BHyOwIh1lZta1+Yb0PaV7xC3ImbIdmwpYxJGLKTAWUK6AorBJIPPERaxGMTCmnO7UhLslguef92sIPzXs6WEl8sKf9W8/sti/iVpliH+aDH/grhd7Ocnv1nFVc0Oq74l2vT1xTLeXS4xzGm7jOomkGFh6J2pxTS+aWrrvUQguyqySAsIWnArsCcRA7XwjFqcgFqARk3BewxLPWop4IcEJmXtCh3oxkD8EHiPAfr8CtQC1JAc1AL0+SO4IsG5IhC0AI1awkEtvEhNKKjHwHx+iixWgNS6yT0+sP8gbDdn1XS7XfS7k0lJ3rdXbfcm9jYNZv16Fcfp5l5/4u5LXtRXk6YUdQTXAZO0V+0FGE/AiCiOAcZLiYmhyvy0MVoDjJeqjHevytY4ZKByPXFprwDRCZV1D3xT1E5fg4pqN6QEFT+oJKDiowaDwfcTjFQA4yWYy2OXg4MDsLkNLzlGwO+p8Y/F5UJzcdo1HID9B5btNiJwyzxRZAyKzEtxUQLi4p9PJmD3/Uz0cwlx8Y0KpXylAOC/FvP/WZx82Dzx/5As5l8u5t827z+ybz5ezP+0mH/X0Qx1PO+/MqXVyUag5CElLiFLAVDifN+UMFFvN0SMQZD8pxQnexck1AnuiEhwCJL/3l0MSv5TEjwGJf+9uyiBUfJc3SUSguS/uou2UHdrnnYD1L41Hny8EEJaHsEu+Z4KZ0gMeS5FClLkvTliMEf+J4Y4HigFQElpUPKdUkxpqw7oIXlmiprUE9J3t2dEJuUgLck7P3vjB1qpiCUkH7u7bn96dExMRfLafqMi9gb10wE5PCbj3J7yML0wo745UG95VFRCXAtjZU+qqPN+z15uOTLT3iCvzPgwt5fV9Ce4QDVKpLwiVRkJGem9UMW0vG6pxiv9JF9CVTGmVQRZDUJWWXxVqokN8/heqKJIcA9gk+SqSpg6fwlgQ9HDkboiWK4SvSewsK4dU43aGbSXWlfJFYN1DcMTplekKqiiWkMJhyKu8qrOcCyVEuKqYF+7OHsPavgmuSb6ql6T5JzHUMNBqGFxVeMqk0RphK6BuExXdYTjWFCpQDUMqvyqKeE4olrBZQolblVXzgpLxhMKcQ2B6spqDC+LW23gKkE1DIdJXzl5qCWnV88Kr2l2DbA3qoTZNcQ1RkIiHLBX1sNCWrLQw0FQvXKaSSVUxMj1B2Fdtb6qEna1z5qCagBUadIOXH8yKY8mNZmaqnpmOVzoz3K6vWdvcoquOfsoAdRx1BkQSMvONFTcGQ34GbvSSHQEZeUTENmu6gCQWwfSXpQNQG59Vm17URUAuW0gigGIX0DaT1jg8N4uDZ7A4fUn/NDwr/wy5wmDePjj7XZIA6Zjd1cXsYdnQBiAeGXOE5hzz7xdBIN+ZXiphkX3x79Cvt0zi47HUT6Jh4L18GrFmVivtKL6XdM+5zc/WMw/X5x8sph/vZh/bzeR5tVHi/kX9tWni/k3zYcfrRKrZodVv8ynTekE2uh0TStaXT0DtHymxShohUKLJYqDVjC0VtZ73ictNCTdUbBiqMGAHMIILkZAalCDVkCypW6MFlo07+oNKqAKRKqYgA4Mx2IlGrQCUoNyF1oHBxfbXIHYnqOtGPIVjO2iMdLvISWdEG2FFG0hoRvQwxIBTRiSJuTwC8PShjKCfAUkXxrpJ19R/T9/jPaKzwMDAA=='

train_file.write_bytes(gzip.decompress(base64.b64decode(EMBEDDED_TRAIN)))
val_file.write_bytes(gzip.decompress(base64.b64decode(EMBEDDED_VAL)))

def get_sha256(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_TRAIN_SHA = 'f67b50c7724e8cd53d029058c688e712b99b9a3f1be3df0a92c756975220fe1c'
EXPECTED_VAL_SHA = 'faa98cf83cbe209736d8890a902f65fa382cb2ff31b96e55608d98d9e68fa29a'

act_train_sha = get_sha256(train_file)
act_val_sha = get_sha256(val_file)

print(f'Train split ({train_file.name}): {act_train_sha} -> {"VERIFIED" if act_train_sha == EXPECTED_TRAIN_SHA else "MISMATCH"}')
print(f'Val split   ({val_file.name}):   {act_val_sha} -> {"VERIFIED" if act_val_sha == EXPECTED_VAL_SHA else "MISMATCH"}')

assert act_train_sha == EXPECTED_TRAIN_SHA, 'Train split checksum mismatch!'
assert act_val_sha == EXPECTED_VAL_SHA, 'Validation split checksum mismatch!'

with open(train_file, 'r', encoding='utf-8') as f:
    train_records = [json.loads(line) for line in f if line.strip()]
with open(val_file, 'r', encoding='utf-8') as f:
    val_records = [json.loads(line) for line in f if line.strip()]

print(f'\nLoaded Clean Splits:')
print(f'  Train records:      {len(train_records):,}')
print(f'  Validation records: {len(val_records):,}')
print(f'  Test split:         QUARANTINED (Zero exposure to model training and selection)')

## 4. Prompt Formatter & Completion-Only Loss Masking Collator
Applies native Qwen ChatML formatting (`<|im_start|>...<|im_end|>`).
Configures label masking so that prompt tokens and padding receive `-100` (ignored by cross-entropy loss), focusing gradients strictly on the target JSON completion.

In [ ]:
from typing import Any, Dict, List
from transformers import AutoTokenizer, AutoConfig
from datasets import Dataset

BASE_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
SYSTEM_PROMPT = (
    'You are NEXUS-SLM, an enterprise customer support intelligence triage agent. '
    'Analyze the customer support ticket and classify it into the predefined taxonomy. '
    'Return valid JSON only.'
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def format_chatml_prompt(ticket_text: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': ticket_text.strip()},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def prepare_masked_dataset(records: List[Dict[str, Any]], max_len: int = 384) -> Dataset:
    input_ids_list = []
    labels_list = []
    for r in records:
        target = {
            'category': r['category'],
            'priority': r['priority'],
            'language': r['language'],
            'requires_human': r['requires_human'],
        }
        completion_str = f"{json.dumps(target, ensure_ascii=False)}<|im_end|>"
        prompt_str = format_chatml_prompt(r['text'])
        full_str = f"{prompt_str}{completion_str}"
        
        prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
        full_ids = tokenizer.encode(full_str, add_special_tokens=False)
        
        if full_ids[:len(prompt_ids)] == prompt_ids:
            p_len = len(prompt_ids)
        else:
            comp_ids = tokenizer.encode(completion_str, add_special_tokens=False)
            p_len = max(0, len(full_ids) - len(comp_ids))
            
        lbls = [-100] * p_len + full_ids[p_len:]
        input_ids_list.append(full_ids[:max_len])
        labels_list.append(lbls[:max_len])
        
    return Dataset.from_dict({'input_ids': input_ids_list, 'labels': labels_list})

class DynamicCompletionCollator:
    def __init__(self, pad_id: int, max_len: int = 384):
        self.pad_id = pad_id
        self.max_len = max_len
    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_b_len = min(max(len(x['input_ids']) for x in batch), self.max_len)
        p_ids, p_mask, p_lbls = [], [], []
        for x in batch:
            ids = x['input_ids'][:max_b_len]
            lbl = x['labels'][:max_b_len]
            pad_n = max_b_len - len(ids)
            p_ids.append(ids + [self.pad_id] * pad_n)
            p_mask.append([1] * len(ids) + [0] * pad_n)
            p_lbls.append(lbl + [-100] * pad_n)
        return {
            'input_ids': torch.tensor(p_ids, dtype=torch.long),
            'attention_mask': torch.tensor(p_mask, dtype=torch.long),
            'labels': torch.tensor(p_lbls, dtype=torch.long),
        }

train_dataset = prepare_masked_dataset(train_records)
val_dataset = prepare_masked_dataset(val_records)
data_collator = DynamicCompletionCollator(pad_id=tokenizer.pad_token_id)

print(f'Prepared Train Dataset:      {len(train_dataset):,} samples')
print(f'Prepared Validation Dataset: {len(val_dataset):,} samples')
print('Sample verification: label tokens trained on:', sum(1 for x in train_dataset[0]['labels'] if x != -100))

## 5. Model Loading & LoRA PEFT Configuration
Initializes `Qwen2.5-0.5B-Instruct` and attaches low-rank adapters ($r=16, \alpha=32$) across all attention and MLP projection layers.

In [ ]:
from transformers import AutoModelForCausalLM, set_seed
from peft import LoraConfig, get_peft_model, TaskType

set_seed(42)
torch.manual_seed(42)

print(f'Loading base model: {BASE_MODEL_ID}...')
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=dtype,
    device_map='auto',
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 6. Supervised LoRA Fine-Tuning
Executes 2 training epochs with cosine learning rate schedule, gradient accumulation, and evaluation per epoch.
Best checkpoint is automatically selected on validation loss.

In [ ]:
import inspect
from transformers import Trainer, TrainingArguments

OUTPUT_DIR = WORK_DIR / 'models' / 'qwen25_05b_clean_v2_lora'
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'

training_kwargs = {
    'output_dir': str(CHECKPOINT_DIR),
    'num_train_epochs': 2,
    'per_device_train_batch_size': 4,
    'per_device_eval_batch_size': 4,
    'gradient_accumulation_steps': 2,
    'learning_rate': 2e-4,
    'lr_scheduler_type': 'cosine',
    'weight_decay': 0.01,
    'fp16': (dtype == torch.float16),
    'bf16': (dtype == torch.bfloat16),
    'logging_steps': 25,
    'save_total_limit': 2,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'report_to': 'none',
    'seed': 42,
}

sig = inspect.signature(TrainingArguments.__init__)
if 'eval_strategy' in sig.parameters:
    training_kwargs['eval_strategy'] = 'epoch'
    training_kwargs['save_strategy'] = 'epoch'
else:
    training_kwargs['evaluation_strategy'] = 'epoch'
    training_kwargs['save_strategy'] = 'epoch'

if 'warmup_ratio' in sig.parameters:
    training_kwargs['warmup_ratio'] = 0.05
else:
    training_kwargs['warmup_steps'] = 60

training_args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print('Beginning Qwen2.5-0.5B LoRA fine-tuning on clean template-grouped dataset...')
train_result = trainer.train()
print('Training complete!')
print(f'Train Runtime: {train_result.metrics["train_runtime"]:.1f} seconds')
print(f'Train Loss:    {train_result.metrics.get("train_loss", 0.0):.4f}')

## 7. Validation-Based Model Selection & Triage Evaluation
Evaluates the best checkpoint on the held-out validation set ($N=449$) across all 14 categories and 3 languages.

In [ ]:
import time
import re
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model.eval()
val_predictions = []
json_valid_count = 0

print(f'Evaluating best model checkpoint on Validation Split (N={len(val_records)})...')
t0 = time.perf_counter()
device = getattr(model, 'device', next(model.parameters()).device)

for idx, r in enumerate(val_records, 1):
    prompt = format_chatml_prompt(r['text'])
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    p_len = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][p_len:], skip_special_tokens=True).strip()
    
    parsed = None
    try:
        clean_raw = re.sub(r'^```json\s*', '', raw)
        clean_raw = re.sub(r'\s*```$', '', clean_raw).strip()
        parsed = json.loads(clean_raw)
        json_valid_count += 1
    except Exception:
        parsed = {'category': 'unknown', 'priority': 'unknown', 'language': 'unknown', 'requires_human': True}
        
    val_predictions.append({
        'ground_truth': r,
        'prediction': parsed,
        'raw': raw,
    })
    
    if idx % 100 == 0 or idx == len(val_records):
        print(f'  Processed {idx}/{len(val_records)} validation samples...')

total_val_sec = time.perf_counter() - t0
CATEGORIES = [
    'payment_failure', 'refund_request', 'account_access', 'password_reset',
    'order_cancel', 'order_tracking', 'subscription', 'billing', 'technical_issue',
    'fraud_report', 'delivery_issue', 'account_closure', 'product_complaint', 'general_query'
]

y_true_cat = [v['ground_truth']['category'] for v in val_predictions]
y_pred_cat = [v['prediction'].get('category', 'unknown') for v in val_predictions]

y_true_prio = [v['ground_truth']['priority'] for v in val_predictions]
y_pred_prio = [v['prediction'].get('priority', 'unknown') for v in val_predictions]

y_true_lang = [v['ground_truth']['language'] for v in val_predictions]
y_pred_lang = [v['prediction'].get('language', 'unknown') for v in val_predictions]

cat_acc = accuracy_score(y_true_cat, y_pred_cat)
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true_cat, y_pred_cat, labels=CATEGORIES, average='macro', zero_division=0)
prio_acc = accuracy_score(y_true_prio, y_pred_prio)
lang_acc = accuracy_score(y_true_lang, y_pred_lang)
json_rate = json_valid_count / len(val_records)

print('\n' + '=' * 75)
print('VALIDATION METRICS FOR MODEL SELECTION (N=449):')
print('=' * 75)
print(f'Category Accuracy:    {cat_acc:.4f} ({sum(1 for yt, yp in zip(y_true_cat, y_pred_cat) if yt == yp)}/{len(val_records)})')
print(f'Category Macro-F1:    {macro_f1:.4f}')
print(f'Priority Accuracy:    {prio_acc:.4f}')
print(f'Language ID Accuracy: {lang_acc:.4f}')
print(f'JSON Validity Rate:   {json_rate:.4f}')
print(f'Total Eval Duration:  {total_val_sec:.2f}s ({len(val_records)/total_val_sec:.1f} samples/s)')
print('=' * 75)

## 8. Adapter Verification & Artifact Packaging
Saves adapter, verifies independent reloading from disk, and packages `qwen25_05b_clean_v2_lora.zip`.

In [ ]:
import shutil
from peft import PeftModel

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Saving LoRA adapter to {OUTPUT_DIR}...')
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

# Independent reload verification
print('Verifying adapter reload from disk...')
test_load_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, dtype=dtype, device_map='auto')
test_peft = PeftModel.from_pretrained(test_load_base, str(OUTPUT_DIR))
test_peft.eval()
test_prompt = format_chatml_prompt('Payment debited but order not confirmed')
test_in = tokenizer(test_prompt, return_tensors='pt').to(device)
with torch.no_grad():
    test_out = test_peft.generate(**test_in, max_new_tokens=40, do_sample=False)
test_resp = tokenizer.decode(test_out[0][test_in['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'Adapter Reload Test PASSED! Test prediction: {test_resp.strip()}')
del test_peft, test_load_base
torch.cuda.empty_cache()

metadata = {
    'candidate_name': 'qwen25_05b_clean_v2_lora',
    'base_model': BASE_MODEL_ID,
    'dataset_name': 'dataset_v2_clean_template_grouped',
    'train_records': len(train_records),
    'val_records': len(val_records),
    'train_runtime_sec': round(float(train_result.metrics.get('train_runtime', 0.0)), 2),
    'train_loss': round(float(train_result.metrics.get('train_loss', 0.0)), 4),
    'validation_metrics': {
        'category_accuracy': round(float(cat_acc), 4),
        'category_macro_f1': round(float(macro_f1), 4),
        'priority_accuracy': round(float(prio_acc), 4),
        'language_accuracy': round(float(lang_acc), 4),
        'json_validity_rate': round(float(json_rate), 4),
    },
    'seed': 42,
    'status': 'experimental_frozen_candidate',
}

with open(OUTPUT_DIR / 'candidate_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

zip_path = WORK_DIR / 'qwen25_05b_clean_v2_lora'
shutil.make_archive(str(zip_path), 'zip', str(OUTPUT_DIR))
full_zip_file = f'{zip_path}.zip'
print(f'Candidate archive successfully created: {full_zip_file}')
print(f'Archive Size: {round(os.path.getsize(full_zip_file) / (1024*1024), 2)} MB')

# Ensure root copy on Kaggle environment for direct artifact discovery
if Path('/kaggle/working').exists() and Path(full_zip_file).parent != Path('/kaggle/working'):
    kaggle_root_zip = Path('/kaggle/working/qwen25_05b_clean_v2_lora.zip')
    shutil.copy2(full_zip_file, kaggle_root_zip)
    print(f'Exported archive to Kaggle root directory: {kaggle_root_zip}')

# Attempt automated download if running in Google Colab
try:
    from google.colab import files
    print('Initiating direct browser download in Google Colab...')
    files.download(full_zip_file)
except Exception:
    print(f'Download manually from: {full_zip_file}')

print('Done!')